# Agentic Search

### RESEARCH QUESTION: How is agentic search different from a regulat search tool?

- In zero shot learning (ZSL), an agent recieves a prompt and produces an answer based on static weights of the model
- ZSL relies on dymanic data, so it wouldn't know things like scores from a recent game
- 

...

In [2]:
# libraries
from dotenv import load_dotenv
import os
from tavily import TavilyClient

# load environment variables from .env file
_ = load_dotenv()

# connect
client = TavilyClient(api_key=os.environ.get("TAVILY_API_KEY"))

In [3]:
# run search
result = client.search("What is in Nvidia's new Blackwell GPU?",
                       include_answer=True)

# print the answer
result["answer"]


'The Blackwell GPU features 5th Generation Tensor Cores with FP4 precision, offering up to 3x higher AI performance. It includes advanced architecture for large-scale AI workloads and includes 96GB of DDR7 ECC memory.'

## Regular Search

In [16]:
# choose location (try to change to your own city!)

city = "Boston"

query = f"""
    what is the current weather in {city}?
    Should I travel there today?
    "weather.com"
"""

In [17]:
import requests
from bs4 import BeautifulSoup
from ddgs import DDGS
import re

ddg = DDGS()

def search(query, max_results=6):
    try:
        results = ddg.text(query, max_results=max_results)
        return [i["href"] for i in results]
    except Exception as e:
        print(f"returning previous results due to exception reaching ddg.")
        results = [ # cover case where DDG rate limits due to high deeplearning.ai volume
            "https://weather.com/weather/today/l/USCA0987:1:US",
            "https://weather.com/weather/hourbyhour/l/54f9d8baac32496f6b5497b4bf7a277c3e2e6cc5625de69680e6169e7e38e9a8",
        ]
        return results  


for i in search(query):
    print(i)

https://weather.com/
https://www.holiday-weather.com/boston
https://1weather.today/
https://uk-weather.com/
https://travelpander.com/average-boston-weather-in-june/
https://www.weather.com.au/vic


In [18]:
def scrape_weather_info(url):
    """Scrape content from the given URL"""
    if not url:
        return "Weather information could not be found."
    
    # fetch data
    headers = {'User-Agent': 'Mozilla/5.0'}
    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        return "Failed to retrieve the webpage."

    # parse result
    soup = BeautifulSoup(response.text, 'html.parser')
    return soup


In [19]:
# use DuckDuckGo to find websites and take the first result
url = search(query)[0]

# scrape first wesbsite
soup = scrape_weather_info(url)

print(f"Website: {url}\n\n")
print(str(soup.body)[:50000]) # limit long outputs

Website: https://weather.com/


<body class="inter_3e69cff8-module__PPczxG__className font-sans"><div hidden=""><!--$--><!--/$--></div><script src="/wx-next-web/static/vendor/@mparticle/web-sdk@2.47.1/dist/mparticle.min.js"></script><script>(self.__next_s=self.__next_s||[]).push(["https://weather.com/api/v1/script/dprSdkScript.js",{"id":"dprsdk-script"}])</script><script>(self.__next_s=self.__next_s||[]).push([0,{"children":"\n\t\t\t\t\tif (window.top?.DprSdk) {\n\t\t\t\t\t\tvar w = window.top;\n\t\t\t\t\t\tif (!w.__twcUserId) {\n\t\t\t\t\t\t\tw.__twcUserId = new Promise(function (r) { w.__twcResolveUserId = r; });\n\t\t\t\t\t\t}\n\t\t\t\t\t\tvar getRegisteredUserId = function () { return w.__twcUserId; };\n\t\t\t\t\t\tasync function initDprSdk() {\n\t\t\t\t\t\t\ttry {\n\t\t\t\t\t\t\t\tawait w.DprSdk.init({\n\t\t\t\t\t\t\t\t\tgetApplicationInfo: function () { return { id: 'weather.com', version: '2.0.0' }; },\n\t\t\t\t\t\t\t\t\tgetUserRegime: function () { return 'usa'; },\n\t\t\t\t\t\

In [20]:
# extract text
weather_data = []
for tag in soup.find_all(['h1', 'h2', 'h3', 'p']):
    text = tag.get_text(" ", strip=True)
    weather_data.append(text)

# combine all elements into a single string
weather_data = "\n".join(weather_data)

# remove all spaces from the combined text
weather_data = re.sub(r'\s+', ' ', weather_data)
    
print(f"Website: {url}\n\n")
print(weather_data)

Website: https://weather.com/


Forecasts Radar Explore More Sign in Get Premium Upgrade Feels Like -- H High -- L Low -- Chance of Rain 0% Today's Outlook Daily Forecast Trending Now Sweltering summer nights are quietly destroying your sleep Is Colorado's Aspen Acres Fire about to be a ‘megafire’? 0:31 Crowd smashes through doors for AC units amid heat wave 0:41 Too hot to exercise? Here's how to beat the heat Sweltering summer nights are quietly destroying your sleep Is Colorado's Aspen Acres Fire about to be a ‘megafire’? 0:31 Crowd smashes through doors for AC units amid heat wave 0:41 Too hot to exercise? Here's how to beat the heat Editor's Pick Super Typhoon Bavi slammed Rota with catastrophic Category 5 winds north of Guam Stay Safe Making use of public water fountains in a heat wave 0:33 Product Reviews & Deals 7 ‘cool’ finds to conquer the heat Keeping You Healthy Sweltering summer nights are quietly destroying your sleep The Weather Channel is the world's most accurate forec

This method requires lots of parsing, but the main issue is that the result is not concise for our question.

## Agentic Search Tool

In [21]:
# run search
result = client.search(query, max_results=1)

# print first result
data = result["results"][0]["content"]

print(data)

{'location': {'name': 'Boston', 'region': 'Massachusetts', 'country': 'United States of America', 'lat': 42.3583, 'lon': -71.0603, 'tz_id': 'America/New_York', 'localtime_epoch': 1783357222, 'localtime': '2026-07-06 13:00'}, 'current': {'last_updated_epoch': 1783356300, 'last_updated': '2026-07-06 12:45', 'temp_c': 22.2, 'temp_f': 72.0, 'is_day': 1, 'condition': {'text': 'Partly cloudy', 'icon': '//cdn.weatherapi.com/weather/64x64/day/116.png', 'code': 1003}, 'wind_mph': 5.6, 'wind_kph': 9.0, 'wind_degree': 114, 'wind_dir': 'ESE', 'pressure_mb': 1022.0, 'pressure_in': 30.19, 'precip_mm': 0.0, 'precip_in': 0.0, 'humidity': 64, 'cloud': 75, 'feelslike_c': 24.6, 'feelslike_f': 76.3, 'windchill_c': 23.6, 'windchill_f': 74.5, 'heatindex_c': 25.4, 'heatindex_f': 77.8, 'dewpoint_c': 18.5, 'dewpoint_f': 65.4, 'vis_km': 16.0, 'vis_miles': 9.0, 'uv': 1.8, 'gust_mph': 7.1, 'gust_kph': 11.4, 'will_it_rain': 0, 'chance_of_rain': 7, 'will_it_snow': 0, 'chance_of_snow': 0}}


In [22]:
import json
from pygments import highlight, lexers, formatters

# parse JSON
parsed_json = json.loads(data.replace("'", '"'))

# pretty print JSON with syntax highlighting
formatted_json = json.dumps(parsed_json, indent=4)
colorful_json = highlight(formatted_json,
                          lexers.JsonLexer(),
                          formatters.TerminalFormatter())

print(colorful_json)


{
    "location": {
        "name": "Boston",
        "region": "Massachusetts",
        "country": "United States of America",
        "lat": 42.3583,
        "lon": -71.0603,
        "tz_id": "America/New_York",
        "localtime_epoch": 1783357222,
        "localtime": "2026-07-06 13:00"
    },
    "current": {
        "last_updated_epoch": 1783356300,
        "last_updated": "2026-07-06 12:45",
        "temp_c": 22.2,
        "temp_f": 72.0,
        "is_day": 1,
        "condition": {
            "text": "Partly cloudy",
            "icon": "//cdn.weatherapi.com/weather/64x64/day/116.png",
            "code": 1003
        },
        "wind_mph": 5.6,
        "wind_kph": 9.0,
        "wind_degree": 114,
        "wind_dir": "ESE",
        "pressure_mb": 1022.0,
        "pressure_in": 30.19,
        "precip_mm": 0.0,
        "precip_in": 0.0,
        "humidity": 64,
        "cloud": 75,
        "feelslike_c": 24.6,
        "feelslike_f": 76.3,
        "windchill_c": 23.6,
        "win

This display shows what an agent needs to correctly answer the question. The benefit of an agent is another layer can be added to make this answer more easily readable by a human.